In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../src/")
sys.path.append("/code/src/")

In [ ]:
import json
import open3d as o3d
import cv2
import numpy as np
import matplotlib.pyplot as plt
import open3d
from numpy.typing import NDArray
from typing import List, Optional, Tuple

In [ ]:
from data_processing.utils.processing_utils import get_poses_from_data, get_images_from_data, get_cameras_data
from data_processing.core.image_processing import read_images_data_from_folder, frame_num
from nerfstudio_integration.nerfstudio_integration import create_nerfstudio_dataset_from_data as create_nerfstudio_dataset
from colmap_utils.colmap_conversion import write_colmap_cameras_txt, write_colmap_images_txt, convert_data_to_colmap
# visualize_cameras was removed in the refactoring — use visualization.viser_visualization instead

# Playground

# Visualization redesign

## Helper functions

In [ ]:
def get_approximate_intrinsics(H:int, W:int, hfov_degrees):
    #Approximate value from Horizontal FOV
    fov = np.deg2rad(hfov_degrees)
    fx = (W / 2.0) / np.tan(fov / 2.0)
    fy = fx
    cx = (W - 1) / 2.0
    cy = (H - 1) / 2.0

    return fx, fy, cx, cy

In [ ]:
def get_camera_corners(image_size_wh:Tuple[int], intrinsics:Optional[List[float]]=None, 
                       hfov_degrees:int=60, z:float=-1.0, scale:float=0.2):
        W, H = image_size_wh

        if intrinsics is None:
            fx, fy, cx, cy = get_approximate_intrinsics(H, W, hfov_degrees)
        else:
            fx, fy, cx, cy = intrinsics

        corners_px = np.array([[0, 0],
                            [W-1, 0],
                            [W-1, H-1],
                            [0, H-1]], dtype=np.float64)
        corners_cam = np.zeros((4, 3), dtype=np.float64)
        corners_cam[:, 0] = (corners_px[:, 0] - cx) / fx
        corners_cam[:, 1] = (corners_px[:, 1] - cy) / fy
        corners_cam[:, 2] = z # forward for NeRF/OpenCV-ish camera is -Z

        corners_cam *= scale

        return corners_cam

In [ ]:
def create_line_cylinder(p1, p2, radius=0.01, resolution=20, color=[1, 0, 0]):
    p1 = np.array(p1)
    p2 = np.array(p2)
    direction = p2 - p1
    length = np.linalg.norm(direction)

    cylinder = o3d.geometry.TriangleMesh.create_cylinder(radius, length, resolution)
    cylinder.paint_uniform_color(color)

    # Align cylinder with direction
    z = np.array([0, 0, 1])
    direction /= length
    v = np.cross(z, direction)
    c = np.dot(z, direction)

    if np.linalg.norm(v) != 0:
        vx = np.array([[0, -v[2], v[1]],
                       [v[2], 0, -v[0]],
                       [-v[1], v[0], 0]])
        R = np.eye(3) + vx + vx @ vx * ((1 - c) / (np.linalg.norm(v) ** 2))
        cylinder.rotate(R, center=np.zeros(3))

    cylinder.translate((p1 + p2) / 2)
    return cylinder

In [ ]:
def create_textured_image_front_and_back(
    image_bgr:NDArray, intrinsics:Optional[List[float]]=None, hfov_degrees:int=60,
    image_size_wh:Tuple[int]=(800, 600), scale:float=0.2, z:float=-1.0,
    c2w:Optional[NDArray]=None, front=True
)->List[o3d.geometry.TriangleMesh]:
    images = []
    image_mesh_front = create_textured_image_plane(image_bgr=image_bgr, intrinsics=intrinsics, hfov_degrees=hfov_degrees,
                                                   image_size_wh=image_size_wh, scale=scale, z=z, c2w=c2w,
                                                   front=True)
    images += image_mesh_front
    image_mesh_back = create_textured_image_plane(image_bgr=image_bgr, intrinsics=intrinsics, hfov_degrees=hfov_degrees,
                                                   image_size_wh=image_size_wh, scale=scale, z=z, c2w=c2w,
                                                   front=False)
    
    images += image_mesh_back
    
    return images


In [ ]:
def create_textured_image_plane(
    image_bgr:NDArray, intrinsics:Optional[List[float]]=None, hfov_degrees:int=60,
    image_size_wh:Tuple[int]=(800, 600), scale:float=0.2, z:float=-1.0,
    c2w:Optional[NDArray]=None, front=True
)->List[o3d.geometry.TriangleMesh]:

    W, H = image_size_wh

    image_bgr = cv2.resize(image_bgr, (W, H), interpolation=cv2.INTER_AREA)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    corners_cam = get_camera_corners(image_size_wh=image_size_wh, intrinsics=intrinsics, 
                                     hfov_degrees=hfov_degrees, z=z, scale=scale)

    if front:
        # showing the image to the front of the plane
        triangles = np.array([
            [0, 1, 2],
            [0, 2, 3],
        ], dtype=np.int32)
    else:
        triangles = np.array([
            [0, 2, 1],
            [0, 3, 2],
        ], dtype=np.int32)

    if front:
        uvs = np.array([
                [0.0, 1.0],
                [1.0, 1.0],
                [1.0, 0.0],

                [0.0, 1.0],
                [1.0, 0.0],
                [0.0, 0.0],
            ], dtype=np.float64)
    else:
        uvs = np.array([
                [0.0, 1.0],
                [1.0, 0.0],
                [1.0, 1.0],

                [0.0, 1.0],
                [0.0, 0.0],
                [1.0, 0.0],
                
            ], dtype=np.float64)


    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(corners_cam)
    mesh.triangles = o3d.utility.Vector3iVector(triangles)
    mesh.triangle_uvs = o3d.utility.Vector2dVector(uvs)
    mesh.triangle_material_ids = o3d.utility.IntVector([0, 0])
    mesh.textures = [o3d.geometry.Image(image_rgb)]
    mesh.compute_vertex_normals()

    if c2w is None:
        c2w = np.eye(4)

    mesh.transform(c2w)
    return [mesh]

In [ ]:
def create_camera_frustum_lines(intrinsics:Optional[List[float]]=None, hfov_degrees:int=60,
                             image_size_wh:Tuple[int]=(800, 600), scale:float=0.2,
                             c2w:Optional[NDArray]=None, color=[0.1, 0.7, 1.0], z=-1.0)->List:

    corners_cam = get_camera_corners(image_size_wh=image_size_wh, intrinsics=intrinsics, 
                                     hfov_degrees=hfov_degrees, z=z, scale=scale)

    orgin = np.zeros((1, 3), dtype=np.float64)
    pts = np.vstack([orgin, corners_cam])

    lines = np.array([[0, 1], [0, 2], [0, 3], [0, 4],
                      [1, 2], [2, 3], [3, 4], [4, 1]], dtype=np.int32)
    
    colors = np.tile(np.array(color), (lines.shape[0], 1))

    ls = o3d.geometry.LineSet()
    ls.points= o3d.utility.Vector3dVector(pts)
    ls.lines = o3d.utility.Vector2iVector(lines)
    ls.colors = o3d.utility.Vector3dVector(colors)

    if c2w is not None:
        ls.transform(c2w)
    return [ls]

In [ ]:
def create_camera_trajectory(camera_centers, color=[0.1, 0.7, 1.0], 
                             radius=0.003, resolution=20)->List[o3d.geometry.TriangleMesh]:
    num_cameras = len(camera_centers)

    lines = []

    for i in range(num_cameras):
        # see if there is a next camera to draw line to
        if i+1 < num_cameras:
            cyl_line = create_line_cylinder(p1=camera_centers[i], p2=camera_centers[i+1], 
                                            radius=radius, resolution=resolution, color=color)
            lines.append(cyl_line)
    return lines

In [ ]:
def create_camera_center_diffrence(camera_center_1, camera_center_2, color=[1, 0, 0], 
                                   radius=0.01, resolution=20)->List[o3d.geometry.TriangleMesh]:

    cyl_line = create_line_cylinder(p1=camera_center_1, p2=camera_center_2, 
                                    radius=radius, resolution=resolution, color=color)
    
    return [cyl_line]

In [ ]:
def create_camera_frustum_cyl(intrinsics:Optional[List[float]]=None, hfov_degrees:int=60,
                             image_size_wh:Tuple[int]=(800, 600), scale:float=0.2,
                             c2w:Optional[NDArray]=None, color=[0.1, 0.7, 1.0], z=-1.0,
                             radius=0.01, resolution=20, show_cam_coord=True, cam_coord_size=0.2)->List[o3d.geometry.TriangleMesh]:

    corners_cam = get_camera_corners(image_size_wh=image_size_wh, intrinsics=intrinsics, 
                                     hfov_degrees=hfov_degrees, z=z, scale=scale)

    orgin = np.zeros((1, 3), dtype=np.float64)
    pts = np.vstack([orgin, corners_cam])
    # pts_homogenious = np.concat([pts, np.ones((pts.shape[0], 1))], axis=1)

    # pts_homogenious = pts_homogenious @ np.linalg.inv(c2w)
    # pts = pts_homogenious[:, :3]

    lines = np.array([[0, 1], [0, 2], [0, 3], [0, 4],
                      [1, 2], [2, 3], [3, 4], [4, 1]], dtype=np.int32)

    ls = []

    for line in lines:
        cyl_line = create_line_cylinder(p1=pts[line[0]], p2=pts[line[1]], 
                                        radius=radius, resolution=resolution, color=color)
        cyl_line.transform(c2w)
        ls.append(cyl_line)

    if show_cam_coord:
        frame_coord = o3d.geometry.TriangleMesh.create_coordinate_frame(size=cam_coord_size)
        frame_coord.transform(c2w)

        ls += [frame_coord]

    return ls

In [ ]:
def create_point_cloud(point_cloud:str)->list:
    pc = o3d.io.read_point_cloud(point_cloud)
    return [pc]

In [ ]:
def get_traj_frames_data(scene_traj_data:List, trajectory_name:str, 
                         cam_intrinsics_type:str="COLMAP", c2w_pose_type:str="COLPMAP"):
    traj_data = scene_traj_data[trajectory_name]
    if cam_intrinsics_type == "COLMAP":
        cam_intrinsics = traj_data["camera_intrinsic_colmap"]
    elif cam_intrinsics_type == "Calibration":
        cam_intrinsics = traj_data["camera_intrinsic_calibration"]
    else:
        raise ValueError(f"Unsupported camera intrinsics type {cam_intrinsics_type}")

    frames = traj_data["frames"]

    if c2w_pose_type == "COLMAP":
        c2w_key = "colmap_pose_c2w"
    elif c2w_pose_type == "Measured":
        c2w_key = "measured_pose_c2w"
    else:
        raise ValueError(f"Unsupported c2w pose type {c2w_pose_type}")

    loaded_frames = []

    for frame in frames:
        if c2w_key not in frame:
            continue

        loaded_frame = {"file_name":frame["file_name"],
                        "pose_c2w":frame[c2w_key],
                        "intrinsics":cam_intrinsics}
        loaded_frames.append(loaded_frame)

    return loaded_frames


In [ ]:
def create_traj_camera_path_vis(poses, images, color=[0,1,0],
                                furstum_line_thickness=0.01, traj_line_thickness=0.005,
                                show_camera_coord=True, scale=1, step=5, show_cam_traj=True):
    pcds = []

    if len(poses) != len(images):
        raise ValueError(f"The number of poses = {len(poses)} is not equal the number of images {len(images)}")

    for i in range(0, len(poses), step):
        pose = poses[i]
        image = images[i]

        H, W = image.shape[:2]

        image_mesh = create_textured_image_front_and_back(image, 
                                                          image_size_wh=(W, H),
                                                          c2w=pose, scale=scale)

        pcds += image_mesh

        w_coord = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1)

        pcds.append(w_coord)

        frustum = create_camera_frustum_cyl(image_size_wh=(W, H), c2w=pose, 
                                            scale=scale, radius=furstum_line_thickness, 
                                            show_cam_coord=show_camera_coord,
                                            color=color)
        
        pcds+=frustum
    if show_cam_traj:    
        pcds += create_camera_trajectory([pose[:3, 3] for pose in poses], radius=traj_line_thickness, color=color)

    return pcds


In [ ]:
def get_traj_camera_centers_pairs(scene_traj_data, traj_name, step=5):
    frames = scene_traj_data[traj_name]["frames"]

    camera_centers_measured = []
    camera_centers_colmap = []
    for i in range(0, len(frames), step):
        frame = frames[i]
        if "colmap_pose_c2w" not in frame:
            continue
        else:
            camera_centers_measured.append(np.array(frame["measured_pose_c2w"])[:3, 3])
            camera_centers_colmap.append(np.array(frame["colmap_pose_c2w"])[:3, 3])
    
    return camera_centers_measured, camera_centers_colmap

In [ ]:
def create_traj_vis(scene_data, trajectory_name, traj_color, step=5, show_cam_traj=True):

    cam_intrinsics_type = "COLMAP"
    c2w_pose_type = "COLMAP"
    traj_frames_colmap = get_traj_frames_data(scene_data["trajectories"], trajectory_name=trajectory_name, 
                            cam_intrinsics_type=cam_intrinsics_type, c2w_pose_type=c2w_pose_type)
    
    poses_colmap = get_poses_from_data(traj_frames_colmap)
    images_colmap = get_images_from_data(traj_frames_colmap, scene_processed_dir)
    
    cam_intrinsics_type = "Calibration"
    c2w_pose_type = "Measured"

    traj_frames_measured = get_traj_frames_data(scene_data["trajectories"], trajectory_name=trajectory_name, 
                            cam_intrinsics_type=cam_intrinsics_type, c2w_pose_type=c2w_pose_type)
    
    poses_measured = get_poses_from_data(traj_frames_measured)
    images_measured = get_images_from_data(traj_frames_measured, scene_processed_dir)

    # create the camera path for measured poses
    pcds_measured_path = create_traj_camera_path_vis(poses_measured, images_measured, 
                                                     color=traj_color, furstum_line_thickness=0.002, 
                                                     traj_line_thickness=0.006, step=step, show_cam_traj=show_cam_traj)

    # create the camera path for colmap poses
    pcds_colmap_path = create_traj_camera_path_vis(poses_colmap, images_colmap,
                                                   color=traj_color, furstum_line_thickness=0.007, 
                                                   traj_line_thickness=0.001, step=step, show_cam_traj=show_cam_traj)
    
    camera_centers_1, camera_centers_2 = get_traj_camera_centers_pairs(scene_data["trajectories"], traj_name=trajectory_name, step=step)

    pcds_camera_center_errors = []
    for camera1, camera2 in zip(camera_centers_1, camera_centers_2):
        pcds_camera_center_errors += create_camera_center_diffrence(camera_center_1=camera1,
                                                                    camera_center_2=camera2,
                                                                    color=[1, 0, 0], radius=0.007)
    
    return pcds_measured_path + pcds_colmap_path + pcds_camera_center_errors

## Test Code

##  Scene visualization

In [ ]:
scene_processed_dir = "G:/Mary/Picture/drone/processed/backyard_scene/"
# scene_processed_dir = "/workspace/datasets/backyard_scene"
scene_processed_json = f"{scene_processed_dir}/scene_data.json"
scene_colmap = f"{scene_processed_dir}/PYCOLMAP_soft_prior"

traj_list = ["traj_1", "traj_6"]
# traj_list = None

In [ ]:
with open(scene_processed_json, "r") as f:
    scene_data = json.load(f)

In [ ]:
if traj_list is None:
    traj_list = list(scene_data["trajectories"].keys())
traj_vis = []

for trajectory_name in traj_list:
    print(trajectory_name)
    images_per_traj_percent = 5

    traj_type = scene_data["trajectories"][trajectory_name]["source_type"]
    traj_color = scene_data["trajectories"][trajectory_name]["color_value"]
    traj_color = [color_comp / 255 for color_comp in traj_color]
    traj_color_name = scene_data["trajectories"][trajectory_name]["color_name"]

    traj_num_frames = scene_data["trajectories"][trajectory_name]["number_frames_in_traj"]

    images_per_traj = int(traj_num_frames * (images_per_traj_percent/100))
    if images_per_traj == 0:
        images_per_traj = 2
    step = int(traj_num_frames/images_per_traj)

    if traj_type == "images":
        show_traj = False
    else:
        show_traj = True
    
    traj_vis += create_traj_vis(scene_data=scene_data, 
                                trajectory_name=trajectory_name, 
                                traj_color=traj_color, 
                                show_cam_traj=show_traj,
                                step=step)
    
point_cloud = scene_processed_dir + scene_data["pointcloud"]
pc = create_point_cloud(point_cloud)

In [ ]:
o3d.visualization.draw_geometries(traj_vis+pc)